# Environment

In [ ]:
import os
os.chdir("..")

In [ ]:
import pandas as pd
import torch
from sklearn.metrics import classification_report
from transformers import BertForSequenceClassification, Trainer, TrainingArguments

from src.models.bert import BertForMultiTaskClassification
from src.pipeline.bert import PipelineBertBinary, PipelineBertMultilabel, BertPipelineConfig
from src.utils.path import DATA_PATH
from src.utils.report import multilabel_report

# Binary Classification on ITED-BR

## Load Model

In [ ]:
config = BertPipelineConfig(
    model_path="./models/bert/binary",
    bert_model="./models/bert/binary",
    label_columns=["offensive"]
)
pipeline = PipelineBertBinary(config)

model = BertForSequenceClassification.from_pretrained("./models/bert/binary")
trainer = Trainer(model=model, args=TrainingArguments(output_dir="./tmp", report_to="none"))

## Load Data

In [ ]:
df = pd.read_csv(f"{DATA_PATH}/ited-br/processed/ited-br.csv")

## Predict

In [ ]:
y_pred = pipeline.predict(trainer, df, "text")
y_true = df["offensive"].values
print(classification_report(y_true, y_pred))

# Multilabel Classification on ITED-BR

In [ ]:
LABEL_COLUMNS = ["insult", "obscene", "ideology", "lgbtqphobia", "racism", "sexism", "xenophobia"]

## Load Model

In [ ]:
config = BertPipelineConfig(
    model_path="./models/bert/multilabel",
    bert_model="./models/bert/multilabel",
    label_columns=LABEL_COLUMNS
)
pipeline = PipelineBertMultilabel(config)

model = BertForMultiTaskClassification.from_pretrained("./models/bert/multilabel")
trainer = Trainer(model=model, args=TrainingArguments(output_dir="./tmp", report_to="none"))

## Predict

In [ ]:
y_pred = pipeline.predict(trainer, df, "text")
y_true = df[LABEL_COLUMNS].values
multilabel_report(y_true, y_pred, label_names=LABEL_COLUMNS)